In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim

from models import SourceEncoder, TargetEncoder

In [18]:
# Load the tensors
x_cat_tensor = torch.load('../dataset/processed/x_cat_tensor.pt')
x_ratio_tensor = torch.load('../dataset/processed/x_ratio_tensor.pt')
categorical_cardinalities = torch.load('../dataset/processed/cardinalities.pt').tolist()

print("데이터 로드 완료:", x_cat_tensor.shape, ",", x_ratio_tensor.shape, ",", len(categorical_cardinalities))

데이터 로드 완료: torch.Size([14676, 23]) , torch.Size([14676, 4]) , 23


In [7]:
# Define PyTorch DataLoader
class UserDataset(Dataset):
    def __init__(self, x_cat_tensor, x_ratio_tensor):
        self.x_cat = x_cat_tensor
        self.x_ratio = x_ratio_tensor
        
    def __len__(self):
        return len(self.x_cat)
        
    def __getitem__(self, idx):
        return self.x_cat[idx], self.x_ratio[idx]

# Create Dataset and DataLoader with the preprocessed tensors
dataset = UserDataset(x_cat_tensor, x_ratio_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True) # Batch size is adjustable

In [9]:
# Contrastive Loss Function (InfoNCE)
def info_nce_loss(z_NF, z_F, temperature=0.1):
    # Normalize two vectors to calculate cosine similarity easily
    z_NF = F.normalize(z_NF, dim=1)
    z_F = F.normalize(z_F, dim=1)
    
    # Calculate cosine similarity matrix for all pairs within the batch (Batch_size x Batch_size)
    similarity_matrix = torch.matmul(z_NF, z_F.T) / temperature
    
    # Ground truth labels (diagonal elements represent positive pairs)
    labels = torch.arange(z_NF.size(0)).long().to(z_NF.device)
    
    # Use Cross Entropy to maximize probability of diagonal elements (Positive) and minimize others (Negative)
    loss = F.cross_entropy(similarity_matrix, labels)
    return loss

In [17]:
# Set Training Loop

# Initialize models (using classes defined in encoding.ipynb)
source_encoder = SourceEncoder(categorical_cardinalities, embed_dim=16, output_dim=128)
target_encoder = TargetEncoder(input_dim=4, output_dim=128)

# Set up optimizer (training parameters for both encoders simultaneously)
optimizer = optim.Adam(list(source_encoder.parameters()) + list(target_encoder.parameters()), lr=0.001)

num_epochs = 50 # number of training epochs

for epoch in range(num_epochs):
    source_encoder.train()
    target_encoder.train()
    
    total_loss = 0.0
    
    for batch_x_cat, batch_x_ratio in dataloader:
        optimizer.zero_grad() # initialize gradients
        
        # pass through models
        z_NF = source_encoder(batch_x_cat)
        z_F = target_encoder(batch_x_ratio)
        
        # Calculate Loss
        loss = info_nce_loss(z_NF, z_F, temperature=0.1)
        
        # Backward pass and weight update
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Contrastive Learning Completed!")

# Save trained model weights (for later use in Stage 3)
torch.save(source_encoder.state_dict(), '../checkpoints/source_encoder.pth')
torch.save(target_encoder.state_dict(), '../checkpoints/target_encoder.pth')

Epoch [1/50], Loss: 5.3700
Epoch [2/50], Loss: 5.2952
Epoch [3/50], Loss: 5.2824
Epoch [4/50], Loss: 5.2694
Epoch [5/50], Loss: 5.2613
Epoch [6/50], Loss: 5.2520
Epoch [7/50], Loss: 5.2466
Epoch [8/50], Loss: 5.2425
Epoch [9/50], Loss: 5.2324
Epoch [10/50], Loss: 5.2325
Epoch [11/50], Loss: 5.2247
Epoch [12/50], Loss: 5.2218
Epoch [13/50], Loss: 5.2138
Epoch [14/50], Loss: 5.2171
Epoch [15/50], Loss: 5.2070
Epoch [16/50], Loss: 5.2023
Epoch [17/50], Loss: 5.2016
Epoch [18/50], Loss: 5.1959
Epoch [19/50], Loss: 5.1925
Epoch [20/50], Loss: 5.1959
Epoch [21/50], Loss: 5.1779
Epoch [22/50], Loss: 5.1825
Epoch [23/50], Loss: 5.1807
Epoch [24/50], Loss: 5.1762
Epoch [25/50], Loss: 5.1673
Epoch [26/50], Loss: 5.1652
Epoch [27/50], Loss: 5.1609
Epoch [28/50], Loss: 5.1603
Epoch [29/50], Loss: 5.1578
Epoch [30/50], Loss: 5.1501
Epoch [31/50], Loss: 5.1487
Epoch [32/50], Loss: 5.1464
Epoch [33/50], Loss: 5.1433
Epoch [34/50], Loss: 5.1392
Epoch [35/50], Loss: 5.1378
Epoch [36/50], Loss: 5.1353
E